In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

print("All imports successful!")

All imports successful!


In [2]:
notes_data = [
    ("Patient reports persistent chest tightness and shortness of breath during exertion.", "Cardiac"),
    ("Follow-up shows blood pressure well controlled on current medication regimen.", "Cardiac"),
    ("Complains of irregular heartbeat and occasional dizziness, recommend ECG.", "Cardiac"),
    ("Patient experiencing palpitations after caffeine intake, otherwise stable.", "Cardiac"),
    ("Persistent dry cough for two weeks, no fever, possible post-viral irritation.", "Respiratory"),
    ("Wheezing noted on examination, history of asthma, prescribed inhaler refill.", "Respiratory"),
    ("Shortness of breath worsened overnight, chest X-ray ordered to rule out pneumonia.", "Respiratory"),
    ("Mild seasonal cough, no other respiratory symptoms reported.", "Respiratory"),
    ("Fasting glucose elevated at last visit, advised dietary changes and re-test.", "Diabetes"),
    ("Patient reports increased thirst and frequent urination over the past month.", "Diabetes"),
    ("HbA1c improved since last checkup, continuing current metformin dosage.", "Diabetes"),
    ("Numbness in feet reported, possible early diabetic neuropathy, referred to specialist.", "Diabetes"),
    ("Patient here for routine annual physical, no acute complaints.", "General"),
    ("Mild seasonal allergies, recommends over-the-counter antihistamine.", "General"),
    ("Patient requests refill of multivitamin, no new symptoms reported.", "General"),
    ("Minor ankle sprain from weekend activity, advised rest and ice.", "General"),
]

notes_df = pd.DataFrame(notes_data, columns=["note_text", "category"])
print(notes_df["category"].value_counts())
notes_df

category
Cardiac        4
Respiratory    4
Diabetes       4
General        4
Name: count, dtype: int64


,note_text,category
0,Patient reports persistent chest tightness and...,Cardiac
1,Follow-up shows blood pressure well controlled...,Cardiac
2,Complains of irregular heartbeat and occasiona...,Cardiac
3,Patient experiencing palpitations after caffei...,Cardiac
4,"Persistent dry cough for two weeks, no fever, ...",Respiratory
5,"Wheezing noted on examination, history of asth...",Respiratory
6,"Shortness of breath worsened overnight, chest ...",Respiratory
7,"Mild seasonal cough, no other respiratory symp...",Respiratory
8,"Fasting glucose elevated at last visit, advise...",Diabetes
9,Patient reports increased thirst and frequent ...,Diabetes


In [3]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=200)
X_tfidf = vectorizer.fit_transform(notes_df["note_text"])
y_labels = notes_df["category"]

print("TF-IDF matrix shape:", X_tfidf.shape)
print("\nSample important words found:")
print(list(vectorizer.vocabulary_.keys())[:15])

TF-IDF matrix shape: (16, 101)

Sample important words found:
['patient', 'reports', 'persistent', 'chest', 'tightness', 'shortness', 'breath', 'exertion', 'follow', 'shows', 'blood', 'pressure', 'controlled', 'current', 'medication']


In [4]:
X_train_txt, X_test_txt, y_train_txt, y_test_txt = train_test_split(
    X_tfidf, y_labels, test_size=0.25, random_state=42, stratify=y_labels
)

note_classifier = LogisticRegression(max_iter=1000, random_state=42)
note_classifier.fit(X_train_txt, y_train_txt)
y_pred_txt = note_classifier.predict(X_test_txt)

print(classification_report(y_test_txt, y_pred_txt, zero_division=0))

              precision    recall  f1-score   support

     Cardiac       0.00      0.00      0.00       1.0
    Diabetes       0.00      0.00      0.00       1.0
     General       0.00      0.00      0.00       1.0
 Respiratory       0.00      0.00      0.00       1.0

    accuracy                           0.00       4.0
   macro avg       0.00      0.00      0.00       4.0
weighted avg       0.00      0.00      0.00       4.0



In [5]:
new_note = ["Patient has high blood sugar and reports frequent urination and fatigue."]
new_tfidf = vectorizer.transform(new_note)
prediction = note_classifier.predict(new_tfidf)
print("Note:", new_note[0])
print("Predicted Category:", prediction[0])

Note: Patient has high blood sugar and reports frequent urination and fatigue.
Predicted Category: Diabetes


In [6]:
import joblib

joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")
joblib.dump(note_classifier, "../models/note_classifier.pkl")
print("NLP models saved!")

NLP models saved!


In [7]:
import torch
print("Torch version:", torch.__version__)


Torch version: 2.12.1+cpu


In [8]:
from transformers import pipeline

print("Loading summarizer...")

try:
    summarizer = pipeline("text-generation", model="gpt2")
    HF_AVAILABLE = True
    print("Summarizer loaded successfully!")
except Exception as e:
    print(f"Could not load model: {e}")
    HF_AVAILABLE = False

Loading summarizer...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Summarizer loaded successfully!


In [9]:
def generate_summary(condition, probability, top_features):
    risk_level = "high" if probability >= 0.5 else "low"
    features = ", ".join(top_features)
    
    prompt = (
        f"Medical Report: Patient screened for {condition}. "
        f"Risk level: {risk_level} ({probability:.0%} probability). "
        f"Key factors: {features}. "
        f"Doctor's note:"
    )
    
    result = summarizer(
        prompt,
        max_new_tokens=80,
        num_return_sequences=1,
        temperature=0.7,
        do_sample=True,
        pad_token_id=50256
    )
    
    generated = result[0]["generated_text"]
    return generated

# Test karte hain
patient_result = {
    "condition": "Diabetes",
    "probability": 0.78,
    "top_features": ["Glucose", "BMI", "Age"]
}

summary = generate_summary(
    patient_result["condition"],
    patient_result["probability"],
    patient_result["top_features"]
)

print("Generated Summary:")
print(summary)

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'temperature', 'pad_token_id', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanu

Generated Summary:
Medical Report: Patient screened for Diabetes. Risk level: high (78% probability). Key factors: Glucose, BMI, Age. Doctor's note: This study was funded by the National Institute on Diabetes and Digestive and Kidney Diseases, Department of Medicine, Bethesda, MD 20402.

BMI

Blood pressure

Blood pressure was measured using a handheld monitor of a 24-hour period. The score was calculated by dividing the patient's blood pressure by the number of hours the patient had been exercising (normal or high). The


In [10]:
import joblib

# Save HF availability flag
joblib.dump(HF_AVAILABLE, "../models/hf_available.pkl")
print("NLP pipeline complete!")
print(f"TF-IDF Vectorizer: Saved")
print(f"Note Classifier: Saved")
print(f"HuggingFace GPT2: {'Available' if HF_AVAILABLE else 'Not Available'}")

NLP pipeline complete!
TF-IDF Vectorizer: Saved
Note Classifier: Saved
HuggingFace GPT2: Available
